## Análise Exploratória de Dados (EDA)

Importando as bibliotecas:

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

### 1. Análise de lotação

Lendo os dados tratados anteriormente:

In [ ]:
matriz_lb = pd.read_csv('../dados_tratados/matriz_linha_bairro_tratado.csv', encoding='utf-8')
sumario = pd.read_csv('../dados_tratados/sumario_servico_dia_tipo_tratado.csv', encoding='utf-8')
lotacao = pd.read_csv('../dados_tratados/lotacao_onibus_tratado.csv', encoding='utf-8')

Cruzando a tabela de lotação (passageiros) com o sumário (quilometragem/cumprimento), e, em seguida, classificando as linhas com base na região dominante:

In [ ]:
df = lotacao.merge(sumario, on=['servico', 'data'], how='inner')    # merge de lotacao e sumario onde servico e data iguais em ambos
df['servico'] = df['servico'].astype(str)                           # colocar a linha do onibus como string

# para cada linha, pegar a região mais frequente entre todos os bairros dela
regiao_por_linha = (
    matriz_lb[matriz_lb['regiao'] != 'Não identificado']
    .groupby('linha')['regiao']
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
    .rename(columns={'linha': 'servico'}) # renomear só aqui para o merge
)

regiao_por_linha['servico'] = regiao_por_linha['servico'].astype(str)

df = df.merge(regiao_por_linha, on='servico', how='left')           # merge left, encaixa o regiao por linha no df mesmo se não tiver correspondente no região por linha
df['regiao'] = df['regiao'].fillna('Não identificado')              # preenche os nan com nao identificado

Consolidando os dados operacionais em médias e totais mensais por linha e região e calculando a métrica de passageiros por viagem:

In [ ]:
df['data'] = pd.to_datetime(df['data'])
df['mes'] = df['data'].dt.to_period('M')                            # cria nova coluna mês, que só pega o ano e o mês da data (2023-07-25 → 2023-07)
# agrupa pelos os que tem o msm servico(linha), mes e regiao
# agg(coluna=('colunax','método') cria colunas novas usando colunas do df mensal agrupado e fazendo alguma operação, unique conta valores unicos, mean média, sum soma tudo)
df_mensal = df.groupby(['servico', 'mes', 'regiao']).agg(
    passageiros_total=('qtd_passageiros_total', 'sum'),
    viagens_total=('qtd_viagens', 'sum'),
    cumprimento=('perc_km_planejada', 'mean')).reset_index()
df_mensal['passageiros_por_viagem']=(df_mensal['passageiros_total']/df_mensal['viagens_total'].replace(0, None)) # adiciona essa coluna passageiros por viagens usando a coluna do df viagens total que acabou de ser criada

# limpeza simples onde tira outliers absurdos
df_mensal = df_mensal[df_mensal['passageiros_por_viagem'] <= 90]

colunas_mensal = ['servico', 'mes', 'regiao', 'passageiros_por_viagem', 'cumprimento']
df_mensal[colunas_mensal].sort_values('passageiros_por_viagem', ascending=False).head().style.set_caption("Top 5 linhas com mais passageiros por viagem e o mês correspondente").hide(axis='index')

...

In [ ]:
# cria a análise por região aqui. df_mensal.groupby('regiao') junta todas as linhas da mesma região
por_regiao = df_mensal.groupby('regiao').agg(
    linhas=('servico','nunique'),
    passageiros_por_viagem=('passageiros_por_viagem','mean'),
    cumprimento=('cumprimento', 'mean'),
    total_passageiros=('passageiros_total','sum')
).round(1).sort_values('cumprimento').reset_index()                 # tinha transformado regiao em indice, reset index volta o regiao como coluna

sumario['data'] = pd.to_datetime(sumario['data'])

PERIODO=f"{sumario['data'].min().date()} a {sumario['data'].max().date()}"

por_regiao.insert(0, 'periodo', PERIODO)                            # coloca o periodo na coluna 0 (no começo)

...

In [ ]:
ranking = df_mensal.groupby(['servico','regiao']).agg(
    meses=('mes','nunique'),
    cumprimento_medio=('cumprimento','mean'),
    passageiros_por_viagem=('passageiros_por_viagem','mean'),
    total_passageiros=('passageiros_total','sum')
).reset_index()

ranking = ranking[ranking['meses'] >= 10]                                               # amostra de 10 meses pra cima
ranking = ranking.sort_values('cumprimento_medio').head(20).reset_index(drop=True)      # só mostra os 20 primeiros
ranking.index+= 1
ranking.insert(0, 'periodo', PERIODO)
ranking = ranking.round(1)